In [10]:
import subprocess
subprocess.run(["uv", "pip", "install", "biopython"], check=True)

Resolved 2 packages in 169ms
 Downloaded biopython
Prepared 1 package in 247ms
Installed 1 package in 10ms
 + biopython==1.87


CompletedProcess(args=['uv', 'pip', 'install', 'biopython'], returncode=0)

In [11]:
# Fetch ~1000 PubMed abstracts on AI/ML in drug discovery
# Uses NCBI's Entrez API 

from Bio import Entrez
import time
import json
import os


Entrez.email = "marieechristine@gmail.com"  

# Step 1: search PubMed for relevant papers
# We use a query that combines AI/ML terms with drug discovery terms.
# [Title/Abstract] tells PubMed to search those fields specifically.
query = (
    '("machine learning"[Title/Abstract] OR "deep learning"[Title/Abstract] '
    'OR "artificial intelligence"[Title/Abstract] OR "neural network"[Title/Abstract]) '
    'AND ("drug discovery"[Title/Abstract] OR "drug design"[Title/Abstract] '
    'OR "drug development"[Title/Abstract] OR "drug repurposing"[Title/Abstract]) '
    'AND hasabstract[Filter] '
    'AND English[Language]'
)

# esearch returns a list of PubMed IDs (PMIDs) matching the query
search_handle = Entrez.esearch(
    db="pubmed",
    term=query,
    retmax=1000,           # max number of PMIDs to return
    sort="relevance",      # most relevant first
)
search_results = Entrez.read(search_handle)
search_handle.close()

pmid_list = search_results["IdList"]
print(f"Found {len(pmid_list)} PMIDs")

Found 1000 PMIDs


In [12]:
# Step 2: fetch the full records for those PMIDs
# We do this in batches of 200 to be polite to the API and avoid timeouts.

all_records = []
batch_size = 200

for start in range(0, len(pmid_list), batch_size):
    batch = pmid_list[start:start + batch_size]
    print(f"Fetching batch {start} to {start + len(batch)}...")
    
    fetch_handle = Entrez.efetch(
        db="pubmed",
        id=",".join(batch),
        rettype="medline",
        retmode="xml",
    )
    records = Entrez.read(fetch_handle)
    fetch_handle.close()
    
    all_records.extend(records["PubmedArticle"])
    
    # Be polite, wait a beat between batches
    time.sleep(0.5)

print(f"Fetched {len(all_records)} full records")

Fetching batch 0 to 200...
Fetching batch 200 to 400...
Fetching batch 400 to 600...
Fetching batch 600 to 800...
Fetching batch 800 to 1000...
Fetched 1000 full records


In [13]:
# Step 3: pull out the fields we actually want
# Each record is a nested dict, so we grab just the bits we need:
# PMID, title, abstract, authors, journal, year.

papers = []

for record in all_records:
    article = record["MedlineCitation"]["Article"]
    
    # Title
    title = str(article.get("ArticleTitle", ""))
    
    # Abstract — sometimes split into multiple sections (Background, Methods, etc.)
    abstract_parts = article.get("Abstract", {}).get("AbstractText", [])
    abstract = " ".join(str(part) for part in abstract_parts)
    
    # Skip papers without an abstract (shouldn't happen with our filter, but just in case)
    if not abstract.strip():
        continue
    
    # PMID
    pmid = str(record["MedlineCitation"]["PMID"])
    
    # Journal
    journal = str(article.get("Journal", {}).get("Title", ""))
    
    # Year — pulled from the journal's pub date
    pub_date = article.get("Journal", {}).get("JournalIssue", {}).get("PubDate", {})
    year = str(pub_date.get("Year", ""))
    
    # Authors — first three is plenty for display
    author_list = article.get("AuthorList", [])
    authors = []
    for a in author_list[:3]:
        if "LastName" in a and "ForeName" in a:
            authors.append(f"{a['ForeName']} {a['LastName']}")
    authors_str = ", ".join(authors)
    if len(author_list) > 3:
        authors_str += " et al."
    
    papers.append({
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "authors": authors_str,
        "journal": journal,
        "year": year,
    })

print(f"Kept {len(papers)} papers with abstracts")

Kept 1000 papers with abstracts


In [14]:
# Step 4: save to disk as JSON
# This is what your assignment_chat code will load later and embed into ChromaDB.

output_path = "./05_src/assignment_chat/data/pubmed_ai_drug_discovery.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, "w") as f:
    json.dump(papers, f, indent=2)

# Check the file size
size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Saved {len(papers)} papers to {output_path}")
print(f"File size: {size_mb:.2f} MB")

Saved 1000 papers to ./05_src/assignment_chat/data/pubmed_ai_drug_discovery.json
File size: 1.52 MB


In [15]:
# Quick sanity check — peek at one record
papers[0]

{'pmid': '33844136',
 'title': 'Artificial intelligence to deep learning: machine intelligence approach for drug discovery.',
 'abstract': 'Drug designing and development is an important area of research for pharmaceutical companies and chemical scientists. However, low efficacy, off-target delivery, time consumption, and high cost impose a hurdle and challenges that impact drug design and discovery. Further, complex and big data from genomics, proteomics, microarray data, and clinical trials also impose an obstacle in the drug discovery pipeline. Artificial intelligence and machine learning technology play a crucial role in drug discovery and development. In other words, artificial neural networks and deep learning algorithms have modernized the area. Machine learning and deep learning algorithms have been implemented in several drug discovery processes such as peptide synthesis, structure-based virtual screening, ligand-based virtual screening, toxicity prediction, drug monitoring an